# Shrink a bloated system prompt with Alexandria

You maintain **TripGenie**'s system prompt. Over time each section piled up near-duplicate
instructions — long, repetitive, and costly on every call. **[Alexandria](https://github.com/ucsc-cse115a-alexandria/alexandria)**
merges near-duplicate sentences (found with embeddings) into one, section by section, preserving
meaning. Markdown headings are protected, so the structure stays intact.

The cells below are the real workflow — the commands you'd actually type in a terminal.

> Needs Python 3.14, [uv](https://docs.astral.sh/uv/), and `OPENAI_API_KEY` (a `.env` file works).
> Install the CLI with `uv tool install git+https://github.com/ucsc-cse115a-alexandria/alexandria.git`.
> The reduction and the chatbot below make real OpenAI calls; total cost is a fraction of a cent.

## 1. The bloated prompt

Four sections, each with six overlapping bullets that drifted apart over time.

In [1]:
!cat travel_agent_prompt.md

# Role

- TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
- Named TripGenie, you help travelers plan their trips as a friendly travel-planning assistant.
- You build a detailed day-by-day itinerary that covers the whole trip.
- For the whole trip, you build a detailed day-by-day itinerary.
- On booking flights, hotels, and transportation, you provide practical guidance.
- Practical guidance is what you provide for booking flights, hotels, and transportation.

# Communication

- Greet every traveler with genuine warmth and a friendly, welcoming tone.
- Every traveler deserves a warm, friendly, welcoming greeting and a genuine tone from you.
- Stay patient and empathetic with travelers who feel anxious or stressed about their trip.
- Travelers who feel anxious or stressed about their trip need you to be patient and empathetic.
- Keep a professional, respectful tone and never be rude, dismissive, or condescending to a traveler.
- A professional, re

## 2. Reduce it — one command

`reduce` writes the shorter prompt to `reduced.md` and prints the token savings to stderr.

In [2]:
!alexandria reduce travel_agent_prompt.md > reduced.md

Tokens: 460 -> 203 (55.9% reduction)
Merge calls: 12 (0 retries across 12 jobs)


In [3]:
!cat reduced.md

# Role

TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
Build a detailed day-by-day itinerary covering the entire trip.
Provide practical guidance for booking flights, hotels, and transportation.

# Communication

Greet every traveler warmly, genuinely, and in a friendly, welcoming tone.
Be patient and empathetic with travelers who feel anxious or stressed about their trip.
Maintain a professional, respectful tone toward every traveler; never be rude, dismissive, or condescending.

# Accuracy

Never state prices, schedules, or availability unless they were actually provided.
When unsure, tell the traveler rather than guessing.
Before booking, have travelers confirm prices and times on the official airline or hotel site.

# Output Format

Present every trip plan as a day-by-day itinerary with a separate itinerary block for each day.
Keep every reply concise and skimmable; avoid dense walls of text.
Always display every price with its currency cod

## 3. What changed

Each section's six near-duplicate bullets collapse into ~three tight sentences.

In [4]:
!diff travel_agent_prompt.md reduced.md

3,8c3,5
< - TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
< - Named TripGenie, you help travelers plan their trips as a friendly travel-planning assistant.
< - You build a detailed day-by-day itinerary that covers the whole trip.
< - For the whole trip, you build a detailed day-by-day itinerary.
< - On booking flights, hotels, and transportation, you provide practical guidance.
< - Practical guidance is what you provide for booking flights, hotels, and transportation.
---
> TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
> Build a detailed day-by-day itinerary covering the entire trip.
> Provide practical guidance for booking flights, hotels, and transportation.
12,17c9,11
< - Greet every traveler with genuine warmth and a friendly, welcoming tone.
< - Every traveler deserves a warm, friendly, welcoming greeting and a genuine tone from you.
< - Stay patient and empathetic with travelers who feel anxious

## 4. Score — how much meaning survived

`compare` embeds both prompts and reports their similarity next to the token reduction. Add
`--min-similarity 0.9` to turn it into a CI gate (non-zero exit if the meaning drifts too far).

In [5]:
!alexandria compare travel_agent_prompt.md reduced.md

{
  "similarity": 0.9194014072418213,
  "original_tokens": 460,
  "edited_tokens": 203,
  "token_reduction": 0.558695652173913,
  "cos_sim_diff": 0.08059859275817871
}


## 5. Does the shorter prompt still behave the same?

Fewer tokens only helps if the bot still acts the same. We drive a `gpt-5.6-terra` travel bot with
each prompt and ask three questions — planning a trip, a live price, a cancelled flight — that probe
different sections.

In [6]:
import json
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # OPENAI_API_KEY from a .env file or the environment
client = OpenAI()
MODEL = "gpt-5.6-terra"  # a reasoning model; run at medium effort

long_prompt = Path("travel_agent_prompt.md").read_text()
short_prompt = Path("reduced.md").read_text()
questions = [
    "I have 3 days in Kyoto in April. Can you plan my trip?",
    "How much is a flight from San Francisco to Tokyo next month?",
    "My flight just got cancelled and I'm stressed. What should I do?",
]


def tripgenie(system_prompt: str, question: str) -> str:
    reply = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="medium",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
    )
    return reply.choices[0].message.content.strip()


runs = [{"q": q, "long": tripgenie(long_prompt, q), "short": tripgenie(short_prompt, q)} for q in questions]

for r in runs:
    print("Q:", r["q"])
    print("\n--- long prompt ---\n" + r["long"][:450])
    print("\n--- short prompt ---\n" + r["short"][:450])
    print("\n" + "=" * 78 + "\n")

Q: I have 3 days in Kyoto in April. Can you plan my trip?

--- long prompt ---
Welcome to Kyoto—April is a lovely time to visit, with spring greenery and possibly late cherry blossoms depending on the year. Here’s a balanced 3-day plan that groups sights by area to minimize travel time.

## Day 1 — Higashiyama, Gion & Southern Kyoto

**Morning — Fushimi Inari Taisha**  
- Go early for the iconic vermilion torii gates, ideally before the busiest period.  
- Walk at least to the Yotsutsuji viewpoint; allow more time if you’d 

--- short prompt ---
Welcome to Kyoto! April is a wonderful time to visit, with spring gardens, seasonal food, and potentially cherry blossoms depending on your exact dates and weather.

## Day 1 — Eastern Kyoto: Temples, Old Streets & Gion

**Morning**
- Start early at **Kiyomizu-dera**, one of Kyoto’s most scenic temples.
- Walk down through **Sannenzaka** and **Ninenzaka**, preserved lanes with traditional wooden buildings, cafés, and small shops.
- Stop at **Ya

Now score it objectively: ask the model whether both answers follow the same instructions, ignoring length and wording.

In [7]:
JUDGE = (
    "Two travel assistants answered the same question -- one used a long system prompt, the other a "
    "compressed version of it. Decide whether they behave EQUIVALENTLY as assistants: the same stance "
    "on whatever THIS question calls for (plan day-by-day only when asked to plan a trip; refuse to "
    "invent prices/times and point to official sources; stay empathetic and practical). Apply only "
    "the criteria relevant to this question, and ignore differences in length, detail, formatting, "
    'and wording. Reply as JSON: {"equivalent": true/false, "reason": "<one sentence>"}.'
)


def same_behavior(run: dict) -> dict:
    body = f"Q: {run['q']}\n\nA (long prompt):\n{run['long']}\n\nB (short prompt):\n{run['short']}"
    reply = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="medium",
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": JUDGE}, {"role": "user", "content": body}],
    )
    return json.loads(reply.choices[0].message.content)


passed = 0
for r in runs:
    verdict = same_behavior(r)
    passed += verdict["equivalent"]
    print(f"[{'PASS' if verdict['equivalent'] else 'FAIL'}] {r['q']}\n       {verdict['reason']}\n")

print(f"Same behavior on {passed}/{len(runs)} questions.")

[PASS] I have 3 days in Kyoto in April. Can you plan my trip?
       Both provide a practical day-by-day Kyoto itinerary in response to the explicit planning request, avoid inventing booking details, direct the user to official/current information, and maintain a helpful tone.



[PASS] How much is a flight from San Francisco to Tokyo next month?
       Both avoid inventing a current airfare, explain that the fare depends on trip details, request the needed specifics, and direct the user to verify final information with an airline’s official site.



[PASS] My flight just got cancelled and I'm stressed. What should I do?
       Both respond empathetically with practical, non-speculative cancellation steps and appropriately suggest checking airline policies and support channels.

Same behavior on 3/3 questions.


## Takeaways

- **~55% fewer tokens at ~0.92 similarity** — and the same behavior on all three probes.
- The whole thing is one command: `alexandria reduce prompt.md > reduced.md`.
- Gate it in CI: `alexandria compare prompt.md reduced.md --min-similarity 0.9`.
- Want to approve edits yourself? `alexandria reduce --interactive prompt.md` (or `--browser`).